# 📘 Azure Databricks Storage & Unity Catalog Notes

---

## 🔹 Managed Resource Group Storage (Important)

When an Azure Databricks workspace is created, it automatically creates:
- A **managed resource group**
- A **storage account (ADLS Gen2)**

### 📦 Containers created internally:
- `root`
- `meta`
- `logs`
- `jobs`
- `ephemeral`
- `unity-catalog-storage`

### 🚫 Key Limitation:
- This storage is **Databricks-managed**
- Protected by **Deny Assignment**
- ❌ Cannot be accessed directly
- ❌ Cannot be used for:
  - Unity Catalog
  - External tables
  - Mount points

👉 This storage is **only for internal Databricks use**

---

## 🔹 DBFS (Databricks File System)

Command:
```python
display(dbutils.fs.ls("/"))
```
---

# 📘 Azure Databricks Storage & Unity Catalog Notes

## 📁 DBFS (Databricks File System)

### 🔹 What you see:

```python
/databricks-datasets/
/databricks-results/
/Volumes/
```

---

### 🧠 Important:
- DBFS is a **virtual abstraction layer**
- It does **NOT show actual ADLS containers**
- Internal containers like `root`, `meta`, `logs` are **hidden**
- It is only a **logical filesystem view inside Databricks**

---

## 🔹 Unity Catalog Basics

### 🏗️ Object Hierarchy:

```python
Catalog
└── Schema (Database)
└── Tables / Views
```

### 📌 Example:
sales_catalog.europe.customers

---

## 🔹 Creating a Catalog

### ❌ Without storage (may fail):
```sql
CREATE CATALOG my_catalog;
```

Error: **Metastore storage root URL does not exist**

```python
AnalysisException: [RequestId=c01f0bc2-bf67-40de-9eb3-7da4ffc6e3e3 ErrorClass=INVALID_STATE] Metastore storage root URL does not exist. Default Storage is enabled in your account. 
You can use the UI to create a new catalog using Default Storage, or please provide a storage location for the catalog (for example 'CREATE CATALOG myCatalog MANAGED LOCATION '<location-path>')
```

🔍 Why this error occurs
- Unity Catalog metastore has no default storage configured
- Databricks does not know where to physically store data
- So catalog creation fails without storage info

✅ With Managed Location (Correct approach)

```sql
CREATE CATALOG my_catalog
MANAGED LOCATION 'abfss://<container>@<storage-account>.dfs.core.windows.net/';
```

# 🏗️ Recommended Setup (Best Practice)

## ✔️ Step 1: Create your own ADLS Gen2 storage
- Create Storage Account in Azure Data Lake Storage Gen2  
- Create Container (example: `uc-data`)

---

## ✔️ Step 2: Grant access to :contentReference[oaicite:1]{index=1}

Use one of the following authentication methods:

- **Managed Identity (Recommended)**
- **Service Principal**

---

## ✔️ Step 3: Create Unity Catalog properly

```sql
CREATE CATALOG my_catalog
MANAGED LOCATION 'abfss://uc-data@myadls.dfs.core.windows.net/';

# 🚫 Important Clarification (Very Important)

## ❌ Do NOT use Databricks-managed storage:

- root  
- meta  
- logs  
- jobs  
- unity-catalog-storage  

---

## 🧠 Why:

- It is system-controlled  
- Protected by Deny Assignment  
- Not accessible for user workloads  
- Not supported for Unity Catalog usage  

---

# 📊 Quick Summary Table

| Component | Purpose | Access |
|----------|--------|--------|
| Managed ADLS (Databricks RG) | Internal system storage | ❌ No access |
| DBFS | Virtual filesystem view | ✅ Limited access |
| Your ADLS Gen2 | Data storage | ✅ Full control |
| Unity Catalog | Governance layer | ✅ Recommended |

In [0]:
display(dbutils.fs.ls("/"))

# 🧭 Optional Advanced Learning

If you want to go deeper, you can explore:

- External Locations in Unity Catalog
- Storage Credentials (IAM-based access)
- Managed vs External Tables
- Unity Catalog Permissions model
- Data lineage tracking

# Unity Catalog in Databricks – Complete Guide

## 1. What is Unity Catalog?
Unity Catalog is a **centralized data governance solution** in Databricks that manages:
- Data access permissions
- Data discovery
- Data lineage
- Data auditing
- Data sharing

It provides **fine-grained access control** for all data assets in Databricks like:
- Tables
- Views
- Files
- Volumes
- Models
- Functions

Unity Catalog works across **all workspaces** and provides a **single governance layer**.

## 2. What is Metastore?

**The metastore is the top-level container for metadata in Unity Catalog. It registers metadata about data and the permissions that govern access to them. Foa a workspace to use Unity Catalog, it must have a Unity Catalog metastore attached. You should have one metastore for each region in which you have workspaces.**<p>
Unlike Hivde metastore, the Unity Catalog metastore is not a service boundary: it runs in a multi-tenant environment and represents a logical boundary for the segregation of data by region for a given Databricks account.

![image_1774386461977.png](./Images/image_1774386461977.png "image_1774386461977.png")

**DataBricks Architecture**
![image_1774386482610.png](./Images/image_1774386482610.png "image_1774386482610.png")

# 3. Why Unity Catalog?
Before Unity Catalog:
- Each workspace had separate metastore
- Access control was at cluster level
- Hard to manage permissions
- No proper data lineage
- No centralized governance

With Unity Catalog:
- Central governance
- Row level security
- Column level security
- Data lineage
- Data audit logs
- Data sharing
- Managed tables & external tables
- Better security model

---

# 4. Unity Catalog Architecture
Unity Catalog has a **3-level namespace**:


# Managed storage location hierarchy

The level at which you define managed storage in Unity Catalog depends on your preferred data isolation model. Your organization may require that certain types of data be stored within specific accounts or buckets in your cloud tenant.

Unity Catalog gives you the ability to configure managed storage locations at the metastore, catalog, or schema level to satisfy such requirements.

**For example, let's say your organization has a company compliance policy that requires production data relating to human resources to reside in the bucket s3://mycompany-hr-prod. In Unity Catalog, you can achieve this requirement by setting a location on a catalog level, creating a catalog called, for example hr_prod, and assigning the location s3://mycompany-hr-prod/unity-catalog to it. This means that managed tables or volumes created in the hr_prod catalog (for example, using CREATE TABLE hr_prod.default.table …) store their data in s3://mycompany-hr-prod/unity-catalog. Optionally, you can choose to provide schema-level locations to organize data within the hr_prod catalog at a more granular level.**

If storage isolation is not required for some catalogs, you can optionally set a storage location at the metastore level. This location serves as a default location for managed tables and volumes in catalogs and schemas that don't have assigned storage. Typically, however, Databricks recommends that you assign separate managed storage locations for each catalog.

The system evaluates the hierarchy of storage locations from schema to catalog to metastore.

For example, if a table myCatalog.mySchema.myTable is created in my-region-metastore, the table storage location is determined according to the following rule:

- If a location has been provided for mySchema, it will be stored there.
- If not, and a location has been provided on myCatalog, it will be stored there.
- Finally, if no location has been provided on myCatalog, it will be stored in the location associated with the my-region-metastore.

![image_1774460993944.png](./Images/image_1774460993944.png "image_1774460993944.png")

# Metastore Level Storage
- Metastore is attached to the Data Lake
- Catalog is managed
- Schema is managed
- Object is managed


Now, data for our objects[managed tables] will be saved in the Metastore Data Lake

In [0]:
%sql
-- Error If using default storage (without managed location)
-- AnalysisException: [RequestId=c01f0bc2-bf67-40de-9eb3-7da4ffc6e3e3 ErrorClass=INVALID_STATE] Metastore storage root URL does not exist. Default Storage is enabled in your account. You can use the UI to create a new catalog using Default Storage, or please provide a storage location for the catalog (for example 'CREATE CATALOG myCatalog MANAGED LOCATION '<location-path>')

CREATE CATALOG IF NOT EXISTS managed_catalog_02
MANAGED LOCATION 'abfss://metastore@adlsdemodatabrickseus202.dfs.core.windows.net/'
COMMENT "Managed metastore location"; 
-- As I was not able to create without managed location, I will use the default storage.

CREATE SCHEMA IF NOT EXISTS managed_catalog_02.managed_schema;
-- COMMENT <comment>; 


CREATE TABLE IF NOT EXISTS managed_catalog_02.managed_schema.managed_table (
  id INT,
  name STRING,
  value DOUBLE
);
-- COMMENT <comment>;

INSERT INTO managed_catalog_02.managed_schema.managed_table
VALUES (1, 'A', 1.0),
       (2, 'B', 2.0),
       (3, 'C', 3.0);

-- USE managed_catalog.managed_schema;
-- DESCRIBE TABLE managed_table;
-- SELECT * FROM managed_catalog.managed_schema.managed_table;
-- DROP TABLE IF EXISTS managed_catalog.managed_schema.managed_table;
-- DROP SCHEMA IF EXISTS managed_catalog.managed_schema;
-- DROP CATALOG IF EXISTS managed_catalog;




In [0]:
%sql
SELECT * FROM managed_catalog_02.managed_schema.managed_table;

# Catalog Level Storage
- Metastore is attached to the Data Lake
- Catalog is also attached to the Data Lake
- Schema is managed
- Object is managed


Now, data for our objects[managed tables] will be saved in the Catalog Data Lake

In [0]:
%sql
-- Error If using default storage (without managed location)
-- AnalysisException: [RequestId=c01f0bc2-bf67-40de-9eb3-7da4ffc6e3e3 ErrorClass=INVALID_STATE] Metastore storage root URL does not exist. Default Storage is enabled in your account. You can use the UI to create a new catalog using Default Storage, or please provide a storage location for the catalog (for example 'CREATE CATALOG myCatalog MANAGED LOCATION '<location-path>')

CREATE CATALOG IF NOT EXISTS external_catalog_02
MANAGED LOCATION 'abfss://metastore@adlsdemodatabrickseus202.dfs.core.windows.net/catalog'
COMMENT "External Catalog location";

CREATE SCHEMA IF NOT EXISTS external_catalog_02.managed_schema;
-- COMMENT <comment>; 


CREATE TABLE IF NOT EXISTS external_catalog_02.managed_schema.managed_table (
  id INT,
  name STRING,
  value DOUBLE
);
-- COMMENT <comment>;

INSERT INTO external_catalog_02.managed_schema.managed_table
VALUES (1, 'A', 1.0),
       (2, 'B', 2.0),
       (3, 'C', 3.0);

-- USE managed_catalog.managed_schema;
-- DESCRIBE TABLE managed_table;
-- SELECT * FROM managed_catalog.managed_schema.managed_table;
-- DROP TABLE IF EXISTS managed_catalog.managed_schema.managed_table;
-- DROP SCHEMA IF EXISTS managed_catalog.managed_schema;
-- DROP CATALOG IF EXISTS managed_catalog;




In [0]:
%sql
SELECT * FROM external_catalog_02.managed_schema.managed_table;

# Schema Level Storage
- Metastore is attached to the Data Lake
- Catalog is also attached to the Data Lake
- Schema is also attached to the Data Lake
- Object is managed


Now, data for our objects[managed tables] will be saved in the Schema Data Lake

In [0]:
%sql
-- Error If using default storage (without managed location)
-- AnalysisException: [RequestId=c01f0bc2-bf67-40de-9eb3-7da4ffc6e3e3 ErrorClass=INVALID_STATE] Metastore storage root URL does not exist. Default Storage is enabled in your account. You can use the UI to create a new catalog using Default Storage, or please provide a storage location for the catalog (for example 'CREATE CATALOG myCatalog MANAGED LOCATION '<location-path>')

CREATE CATALOG IF NOT EXISTS external_catalog_02
MANAGED LOCATION 'abfss://metastore@adlsdemodatabrickseus202.dfs.core.windows.net/catalog'
COMMENT "External Catalog location";

CREATE SCHEMA IF NOT EXISTS external_catalog_02.external_schema
MANAGED LOCATION 'abfss://metastore@adlsdemodatabrickseus202.dfs.core.windows.net/schema'
COMMENT "External Schema location"; 


CREATE TABLE IF NOT EXISTS external_catalog_02.external_schema.managed_table (
  id INT,
  name STRING,
  value DOUBLE
);
-- COMMENT <comment>;

INSERT INTO external_catalog_02.external_schema.managed_table
VALUES (1, 'A', 1.0),
       (2, 'B', 2.0),
       (3, 'C', 3.0);

-- USE managed_catalog.managed_schema;
-- DESCRIBE TABLE managed_table;
-- SELECT * FROM managed_catalog.managed_schema.managed_table;
-- DROP TABLE IF EXISTS managed_catalog.managed_schema.managed_table;
-- DROP SCHEMA IF EXISTS managed_catalog.managed_schema;
-- DROP CATALOG IF EXISTS managed_catalog;




In [0]:
%sql
SELECT * FROM external_catalog_02.external_schema.managed_table;

# Mount external storage (Best practice)

Instead of using the managed storage, create your own storage account and mount it:
Example (ADLS Gen2):

```python

configs = {
  "fs.azure.account.key.<storage-account>.dfs.core.windows.net": "<access-key>"
}

dbutils.fs.mount(
  source = "abfss://<container>@<storage-account>.dfs.core.windows.net/",
  mount_point = "/mnt/mydata",
  extra_configs = configs)
  
```

# Use Unity Catalog (Modern approach)

If you're using newer Databricks setups:

- Create external locations
- Assign permissions via Unity Catalog
- Access data securely without mounts

# 🚀 Databricks Tables – Complete Interview Cheat Sheet
(Managed vs External + UNDROP + Time Travel)

---

## 1. Managed vs External Tables

### Managed Table

A Managed Table is a table where Databricks manages both **metadata and data**.

- Data stored in Databricks-managed location  
- No LOCATION required  
- DROP TABLE → data deleted (logically, but recoverable for some time)  

Example:

CREATE TABLE sales.bronze.customers (
    id INT,
    name STRING
);

---

### External Table

An External Table is a table where Databricks manages only **metadata**.

- You provide storage location  
- LOCATION is mandatory  
- DROP TABLE → only metadata deleted, data remains  

Example:

CREATE TABLE sales.bronze.customers  
LOCATION 'abfss://raw@storageaccount.dfs.core.windows.net/customers';

---

## 2. Key Differences (Interview Table)

Feature | Managed Table | External Table  
---|---|---  
Data location | Databricks | User  
Metadata | Databricks | Databricks  
DROP TABLE | Deletes data | Keeps data  
LOCATION | Not required | Required  
Control | Less | Full  

---

## 3. UNDROP (Recovery Feature)

### What is UNDROP?

UNDROP allows you to **restore a dropped table**.

Command:

UNDROP TABLE sales.bronze.customers;

---

### Works For:

- Managed Tables  
- Delta Tables (Unity Catalog)  

---

### Does NOT Work For:

- External Tables  

---

### Important Notes:

- Works only within **retention period (~7 days)**  
- After that → permanent deletion  
- Requires Unity Catalog  

---

### Interview Point:

Managed Table → Can UNDROP  
External Table → Cannot UNDROP  

---

## 4. Time Travel (Delta Tables)

Time Travel allows you to **query previous versions of data**.

---

### Query Older Version

SELECT * FROM sales.bronze.customers VERSION AS OF 3;

OR

SELECT * FROM sales.bronze.customers  
TIMESTAMP AS OF '2024-01-01';

---

### Restore Table to Previous Version

RESTORE TABLE sales.bronze.customers TO VERSION AS OF 3;

---

### How it Works:

- Delta Lake stores transaction logs  
- Keeps history of changes  
- Enables rollback and auditing  

---

### Retention:

- Default: **7 days**  
- Controlled by Delta configuration  

---

### Important:

- Works for **Delta Tables only**  
- Works for both Managed and External (if Delta format)  

---

## 5. Drop Behavior Summary

Action | Managed Table | External Table  
---|---|---  
DROP TABLE | Data deleted (recoverable) | Only metadata deleted  
UNDROP | ✅ Yes | ❌ No  
Time Travel | ✅ Yes (Delta) | ✅ Yes (if Delta)  

---

## 6. Main Difference (Very Important for Interviews)

Feature | Managed Table | External Table  
---|---|---  
Data location | Managed by Databricks | Managed by User  
Metadata | Databricks | Databricks  
Data deletion | Deleted when table dropped | Not deleted  
Need LOCATION | No | Yes  
Use case | ETL / Medallion | Existing data lake  
Storage control | No | Yes  

---

## 7. When to Use Managed vs External

### Use Managed Tables When:

- Building Bronze / Silver / Gold tables  
- ETL pipelines  
- Delta tables  
- Databricks manages everything  
- Recommended for Medallion architecture  

### Use External Tables When:

- Data already exists in Data Lake  
- Multiple tools use same data  
- You want full control of storage  
- Sharing data across systems  
- Raw data ingestion  

---

## 8. Real Data Engineering Architecture Example

ADLS  
   |  
   |--- raw/                → External Tables (Bronze Raw Files)  
   |  
   |--- databricks/  
            |  
            |--- bronze tables (Managed)  
            |--- silver tables (Managed)  
            |--- gold tables (Managed)  

### Typical Setup:

- Raw data → External Table  
- Bronze → Managed  
- Silver → Managed  
- Gold → Managed  

👉 This is a very common architecture.

---

## 9. How to Check Table Type

Run:

DESCRIBE DETAIL sales.bronze.customers;

Look at:

- Type: MANAGED or EXTERNAL  
- Location: path  

---

## 8. Final Interview Answer (Perfect 3 Lines)

- Managed Table: Databricks manages both metadata and data, supports UNDROP, and deletes data on drop (recoverable within retention).  
- External Table: Databricks manages only metadata, data remains in external storage, and not deleted when table is dropped. (UNDROP is not supported)
- Time Travel: Allows querying or restoring previous versions of Delta tables using version or timestamp.  

```SQL
-- Remove all data (but keeps table)
TRUNCATE TABLE external_catalog_02.external_schema.managed_table;

-- Restore previous version (works)
RESTORE TABLE external_catalog_02.external_schema.managed_table VERSION AS OF 1;

-- DROP TABLE removes metadata → RESTORE will NOT work
````
---

## 9. Memory Tricks

Managed Table  → Databricks owns data → Can UNDROP  
External Table → You own data → No UNDROP  

Time Travel → “Go back in time” for Delta tables ⏳

In [0]:
%sql
-- DESCRIBE DETAIL external_catalog_02.external_schema.managed_table;
DESCRIBE HISTORY external_catalog_02.managed_schema.managed_table;

In [0]:
%sql
-- We will try to delete the managed table
DROP TABLE IF EXISTS external_catalog_02.external_schema.managed_table;

-- We will UNDROP the table.
UNDROP TABLE external_catalog_02.external_schema.managed_table;

## ❓ Why External Tables Cannot Be UNDROPPED (Even with ADLS Soft Delete)

---

## 1. Key Idea

UNDROP works only when **Databricks controls both:**
- Metadata  
- Data lifecycle  

👉 This is true for **Managed Tables**, NOT External Tables.

---

## 2. What Happens in Managed Tables

When you run:

DROP TABLE sales.bronze.customers;

Databricks:

- Removes table metadata  
- Marks data for deletion (not immediate hard delete)  
- Keeps data + transaction logs for retention period  

So Databricks can:

UNDROP TABLE sales.bronze.customers;

✅ Because it knows:
- Where data is stored  
- Which files belong to the table  
- Full history (Delta logs)  

---

## 3. What Happens in External Tables

When you run:

DROP TABLE sales.bronze.customers;

Databricks:

- Deletes only metadata from Unity Catalog  
- Has **no ownership of data**  
- Does NOT track lifecycle of files  

👉 After drop:
- Databricks "forgets" the table completely  

---

## 4. Where ADLS Soft Delete Fits

Yes — you are correct:

ADLS can have **Soft Delete enabled** ✅

That means:
- Files can be recovered at storage level  

BUT ❗

Databricks does NOT:
- Track ADLS soft delete state  
- Know which files belonged to the table  
- Reconstruct table metadata automatically  

---

## 5. The Core Reason (Most Important)

👉 UNDROP = Metadata + Data Recovery  

For External Tables:

- Metadata ❌ (deleted permanently)  
- Data ✅ (may exist in ADLS)  

Since metadata is gone → Databricks cannot rebuild the table  

---

## 6. Real-World Analogy

Managed Table:
- Like Google Docs  
- Delete → goes to trash → can restore  

External Table:
- Like deleting a shortcut to a file in Drive  
- File may still exist somewhere  
- But shortcut (metadata) is gone  

---

## 7. Can You Still Recover External Table?

YES — but manually:

Step 1: Recover files from ADLS (if soft delete enabled)  
Step 2: Recreate table:

CREATE TABLE sales.bronze.customers  
LOCATION 'abfss://...';

⚠️ But:
- Schema must be known  
- History is lost  
- No automatic UNDROP  

---

## 8. Interview-Perfect Answer

External tables cannot be undropped because Databricks only manages metadata, and once it is deleted, Databricks loses track of the data. Even if ADLS soft delete is enabled, recovery must be done manually since Databricks does not control or track the underlying data lifecycle.

### External Tables OR Table Level Storage

# Object Level Storage [External Tables]
- Metastore is attached to the Data Lake
- Catalog is also attached to the Data Lake
- Schema is also attached to the Data Lake
- Object is also attached to the Data Lake


Now, data for our objects[managed tables] will be saved in the Object Data Lake

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS external_catalog_02
MANAGED LOCATION 'abfss://metastore@adlsdemodatabrickseus202.dfs.core.windows.net/catalog'
COMMENT "External Catalog location";

CREATE SCHEMA IF NOT EXISTS external_catalog_02.external_schema
MANAGED LOCATION 'abfss://metastore@adlsdemodatabrickseus202.dfs.core.windows.net/schema'
COMMENT "External Schema location"; 


CREATE TABLE IF NOT EXISTS external_catalog_02.external_schema.external_table (
  id INT,
  name STRING,
  value DOUBLE
)
LOCATION 'abfss://raw@adlsdemodatabrickseus202.dfs.core.windows.net/external_table'
COMMENT 'External Table location';

INSERT INTO external_catalog_02.external_schema.external_table
VALUES (1, 'A', 1.0),
       (2, 'B', 2.0),
       (3, 'C', 3.0);

In [0]:
%sql
SELECT * FROM external_catalog_02.external_schema.external_table;

In [0]:
%sql
-- We will try to delete the external table
DROP TABLE external_catalog_02.external_schema.external_table;

### Correct Way to Recreate an External Table Using Existing Data

### **1. Notes**
- USING DELTA is optional depending on file format (Delta / Parquet / CSV)
- Databricks will not copy the data; it will just register metadata in Unity Catalog
- The table now points to existing files in ADLS

### **2. Important Tips**
Schema Matching
- If files already exist, you must either:
    - Provide the schema explicitly (recommended for CSV/Parquet)
    - Or use Delta (if Delta format, schema can be inferred automatically)

```SQL

CREATE TABLE external_catalog_02.external_schema.external_table
(
    id INT,
    name STRING,
    amount DOUBLE
)
USING PARQUET
LOCATION 'abfss://raw@storageaccount.dfs.core.windows.net/external_table/';
```

**Note:** If your external table was Delta, you can just run the same CREATE TABLE ... LOCATION command — Delta will read transaction logs and rebuild table metadata.

### **3. Workflow Diagram (Logical)**
```python
ADLS Storage
┌───────────────────────────────┐
│ external_table/               │
│ ├─ part-00000.parquet         │
│ ├─ part-00001.parquet         │
│ └─ _delta_log/                │  <-- Delta only
└───────────────────────────────┘
        ↑
        │ CREATE TABLE external_table LOCATION '...';
        │  -> registers metadata
Databricks/Unity Catalog

```

### **4. Interview-Friendly Explanation**

> When you drop an external table, the data files remain in ADLS. To “re-register” the table without copying files, simply create a new external table pointing to the existing storage location. This allows Databricks to reuse the data and build metadata without duplicating files.

In [0]:
%sql
CREATE TABLE external_catalog_02.external_schema.external_table
USING DELTA
LOCATION 'abfss://raw@adlsdemodatabrickseus202.dfs.core.windows.net/external_table/'

# Volumes

**Databricks Volumes are Unity Catalog objects used for accessing, storing, governing, and organizing non-tabular data files in cloud object storage.** You can use volumes to store and access files in any format, including structured, semi-structured, and unstructured data.

## External Volumes

### Create External Volume

In [0]:
%sql
CREATE EXTERNAL VOLUME external_catalog_02.external_schema.external_volume
LOCATION 'abfss://raw@adlsdemodatabrickseus202.dfs.core.windows.net/external_volume'
COMMENT 'External Volume location';

### Read External Volume from Unity Catalog

In [0]:
%sql
SELECT * FROM CSV.`/Volumes/external_catalog_02/external_schema/external_volume/complex_users.csv`

In [0]:
df = spark.read.format("csv") \
                .option("header","true") \
                .option("inferSchema","true") \
                .load("/Volumes/external_catalog_02/external_schema/external_volume/complex_users.csv")

display(df)